# Coastal flood step 19: negative avoided EAD distance-threshold diagnostics (fixed 5,000 m context)

This notebook performs a **negative-only** distance-threshold diagnostic for assets with `Avoided_EAD_USD < 0`
(increased damages), using nearest-mangrove distances and the same threshold list used previously.

Positive avoided-EAD threshold analysis is intentionally excluded here.


In [ ]:
from pathlib import Path

import numpy
import pandas
import geopandas
import matplotlib.pyplot as plt

pandas.set_option('display.max_columns', 220)
pandas.set_option('display.width', 240)

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)


In [ ]:
# User parameters
SCENARIO_FOR_ATTRIBUTION = 'minimum'  # 'minimum' or 'maximum'
SCENARIOS_FOR_THRESHOLDS = ['minimum', 'maximum']
FIXED_BUFFER_M = 5000.0

THRESHOLDS_M = [250, 500, 1000, 1500, 2000, 3000, 5000, 10000, 15000, 20000, 25000]

MAP_SCENARIO = SCENARIO_FOR_ATTRIBUTION
MAP_THRESHOLD_BINS_M = [250, 500, 1000, 1500, 2000, 3000, 5000, 10000, 15000, 20000, 25000]

if SCENARIO_FOR_ATTRIBUTION not in {'minimum', 'maximum'}:
    raise ValueError("SCENARIO_FOR_ATTRIBUTION must be 'minimum' or 'maximum'.")
for s in SCENARIOS_FOR_THRESHOLDS:
    if s not in {'minimum', 'maximum'}:
        raise ValueError(f'Invalid scenario in SCENARIOS_FOR_THRESHOLDS: {s}')
if MAP_SCENARIO not in SCENARIOS_FOR_THRESHOLDS:
    raise ValueError('MAP_SCENARIO must be included in SCENARIOS_FOR_THRESHOLDS.')

base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
network_csv = base_path / 'dphil_common_cross_cutting/common_incoming_data/networks/network_layers_hazard_intersections_details.csv'
shared_intersections_path = base_path / 'dphil_paper_3/results/01_hazard_infrastructure_network_intersections/coastal_flood_network_intersections'
mangrove_path = base_path / 'dphil_paper_3/inputs/forces_of_nature_mangroves/mangroves.shp'
jamaica_boundary_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'

for p in [network_csv, mangrove_path, jamaica_boundary_path]:
    if not p.exists():
        raise FileNotFoundError(f'Missing required file: {p}')

print(f'Scenario for attribution: {SCENARIO_FOR_ATTRIBUTION}')
print(f'Fixed mangrove buffer (m): {FIXED_BUFFER_M:,.0f}')


In [ ]:
# Load shared inputs
network_details = pandas.read_csv(network_csv)
required_cols = ['asset_gpkg', 'asset_layer', 'asset_description', 'asset_id_column', 'sector']
missing_cols = [c for c in required_cols if c not in network_details.columns]
if missing_cols:
    raise KeyError(f'Missing required columns in network csv: {missing_cols}')

network_details = network_details[required_cols].drop_duplicates().copy()

mangroves = geopandas.read_file(mangrove_path)
if mangroves.crs is None:
    raise ValueError('Mangrove CRS is missing.')
if str(mangroves.crs).upper() != 'EPSG:3448':
    mangroves = mangroves.to_crs('EPSG:3448')

if 'ID' in mangroves.columns:
    mangroves['Mangrove_ID'] = mangroves['ID'].astype(int)
else:
    mangroves['Mangrove_ID'] = numpy.arange(1, len(mangroves) + 1)

mangrove_base_cols = ['Mangrove_ID']
for c in ['Parish', 'HECTARES', 'TYPE']:
    if c in mangroves.columns:
        mangrove_base_cols.append(c)

jamaica_boundary = geopandas.read_file(jamaica_boundary_path)
if jamaica_boundary.crs is None:
    raise ValueError('Jamaica boundary CRS is missing.')
if str(jamaica_boundary.crs).upper() != 'EPSG:3448':
    jamaica_boundary = jamaica_boundary.to_crs('EPSG:3448')

print(f'Mangrove patches: {len(mangroves):,}')
print('Network layers in metadata:', len(network_details))


In [ ]:
# Helper: build one geometry per asset with avoided EAD
asset_key_cols = ['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID']

def build_asset_gdf_for_scenario(scenario):
    results_path = base_path / f'dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_{scenario}_scenario'
    asset_out = results_path / 'damage_estimates' / 'coastal_ead_asset_level_usd.csv'
    if not asset_out.exists():
        raise FileNotFoundError(f'Missing asset-level EAD file: {asset_out}')

    asset_ead = pandas.read_csv(asset_out)

    map_layers = []
    missing_split_files = []

    for row in network_details.itertuples(index=False):
        split_file = shared_intersections_path / f"{row.asset_gpkg}_splits__coastal_flood_rasters_for_intersections__{row.asset_layer}.geoparquet"
        if not split_file.exists():
            missing_split_files.append(str(split_file))
            continue

        split_geom = geopandas.read_parquet(split_file)
        if split_geom.crs is not None:
            split_geom = split_geom.to_crs('EPSG:3448')
        if row.asset_id_column not in split_geom.columns:
            continue

        split_geom = split_geom[[row.asset_id_column, 'geometry']].copy()
        split_geom = geopandas.GeoDataFrame(split_geom, geometry='geometry', crs='EPSG:3448')

        ead_subset = asset_ead.loc[
            (asset_ead['Asset'] == row.asset_gpkg) & (asset_ead['Layer'] == row.asset_layer),
            ['Asset_ID', 'Avoided_EAD_USD']
        ].copy()

        if ead_subset.empty:
            continue

        split_geom['_join_id'] = split_geom[row.asset_id_column].astype(str)
        ead_subset['_join_id'] = ead_subset['Asset_ID'].astype(str)

        merged = split_geom.merge(
            ead_subset[['_join_id', 'Avoided_EAD_USD']],
            on='_join_id',
            how='left'
        )

        merged['Sector'] = row.sector
        merged['Subsector'] = row.asset_description
        merged['Asset'] = row.asset_gpkg
        merged['Layer'] = row.asset_layer
        merged['Asset_ID'] = merged[row.asset_id_column].astype(str)
        merged['Avoided_EAD_USD'] = merged['Avoided_EAD_USD'].fillna(0.0)

        map_layers.append(merged[asset_key_cols + ['Avoided_EAD_USD', 'geometry']])

    if not map_layers:
        raise ValueError(f'No map layers could be built for scenario: {scenario}')

    asset_gdf = geopandas.GeoDataFrame(pandas.concat(map_layers, ignore_index=True), geometry='geometry', crs='EPSG:3448')
    asset_gdf = asset_gdf.dissolve(
        by=asset_key_cols,
        as_index=False,
        aggfunc={'Avoided_EAD_USD': 'first'}
    )
    asset_gdf = geopandas.GeoDataFrame(asset_gdf, geometry='geometry', crs='EPSG:3448')

    out = {
        'results_path': results_path,
        'asset_gdf': asset_gdf,
        'missing_split_files': sorted(set(missing_split_files)),
        'asset_ead_rows': len(asset_ead),
    }
    return out


In [ ]:
# Build scenario asset geodataframes once
scenario_data = {}
for scenario in SCENARIOS_FOR_THRESHOLDS:
    d = build_asset_gdf_for_scenario(scenario)
    scenario_data[scenario] = d
    print(f"{scenario}: assets={len(d['asset_gdf']):,}, EAD rows={d['asset_ead_rows']:,}, missing split files={len(d['missing_split_files'])}")


## Separate check: negative avoided EAD distance thresholds

This is a separate diagnostic for assets with `Avoided_EAD_USD < 0` (increased damages).
It keeps the existing positive-threshold analysis unchanged and writes separate CSV outputs.


In [ ]:
# Distance-threshold calculations (negative avoided EAD assets only)
negative_threshold_tables = {}
negative_bin_tables = {}
negative_nearest_assets_by_scenario = {}

for scenario in SCENARIOS_FOR_THRESHOLDS:
    asset_gdf = scenario_data[scenario]['asset_gdf']
    results_path = scenario_data[scenario]['results_path']

    negative_assets = asset_gdf.loc[asset_gdf['Avoided_EAD_USD'] < 0, asset_key_cols + ['Avoided_EAD_USD', 'geometry']].copy()
    if negative_assets.empty:
        print(f'{scenario}: no assets with negative avoided EAD')
        continue

    nearest_neg = geopandas.sjoin_nearest(
        negative_assets,
        mangroves[['Mangrove_ID', 'geometry']],
        how='left',
        distance_col='nearest_mangrove_distance_m'
    )
    nearest_neg = nearest_neg.dropna(subset=['nearest_mangrove_distance_m']).copy()

    negative_nearest_assets_by_scenario[scenario] = nearest_neg

    total_assets = len(nearest_neg)
    total_negative_abs = float(nearest_neg['Avoided_EAD_USD'].abs().sum())
    total_negative_signed = float(nearest_neg['Avoided_EAD_USD'].sum())

    # Cumulative threshold table
    threshold_rows = []
    for t in THRESHOLDS_M:
        within = nearest_neg.loc[nearest_neg['nearest_mangrove_distance_m'] <= float(t)]
        n = int(len(within))
        a_signed = float(within['Avoided_EAD_USD'].sum())
        a_abs = float(within['Avoided_EAD_USD'].abs().sum())
        threshold_rows.append({
            'Scenario': scenario,
            'Threshold_m': float(t),
            'Assets_within_n': n,
            'Assets_within_pct': 100.0 * n / total_assets if total_assets > 0 else numpy.nan,
            'Negative_Avoided_EAD_within_USD_signed': a_signed,
            'Negative_Avoided_EAD_within_USD_abs': a_abs,
            'Negative_Avoided_EAD_within_abs_pct': 100.0 * a_abs / total_negative_abs if total_negative_abs > 0 else numpy.nan,
        })

    threshold_df = pandas.DataFrame(threshold_rows)
    negative_threshold_tables[scenario] = threshold_df

    # Bin table across the same threshold edges
    max_dist = float(nearest_neg['nearest_mangrove_distance_m'].max())
    bin_edges = [0.0] + [float(t) for t in THRESHOLDS_M] + [float(numpy.ceil(max_dist))]
    clean_edges = [bin_edges[0]]
    for e in bin_edges[1:]:
        if e > clean_edges[-1]:
            clean_edges.append(e)

    bins = pandas.IntervalIndex.from_breaks(clean_edges, closed='right')
    nearest_for_bins = nearest_neg.copy()
    nearest_for_bins['distance_bin'] = pandas.cut(nearest_for_bins['nearest_mangrove_distance_m'], bins=bins)

    bin_df = nearest_for_bins.groupby('distance_bin', observed=False).agg(
        Asset_count=('Asset_ID', 'size'),
        Negative_Avoided_EAD_USD_signed=('Avoided_EAD_USD', 'sum'),
        Negative_Avoided_EAD_USD_abs=('Avoided_EAD_USD', lambda s: s.abs().sum())
    ).reset_index()
    bin_df = bin_df.loc[bin_df['Asset_count'] > 0].copy()
    bin_df['Scenario'] = scenario
    bin_df['Asset_pct'] = 100.0 * bin_df['Asset_count'] / total_assets
    bin_df['Negative_Avoided_EAD_abs_pct'] = 100.0 * bin_df['Negative_Avoided_EAD_USD_abs'] / total_negative_abs if total_negative_abs > 0 else numpy.nan
    bin_df['distance_bin'] = bin_df['distance_bin'].astype(str)
    negative_bin_tables[scenario] = bin_df

    # Save outputs to the same folder, with negative-specific filenames
    out_dir = results_path / 'damage_estimates' / 'mangrove_attribution_fixed_5000m'
    out_dir.mkdir(parents=True, exist_ok=True)
    threshold_df.to_csv(out_dir / 'distance_threshold_distribution_negative_avoided_assets.csv', index=False)
    bin_df.to_csv(out_dir / 'distance_bin_distribution_negative_avoided_assets.csv', index=False)

    print(
        f"{scenario}: negative assets={total_assets:,}, average distance={nearest_neg['nearest_mangrove_distance_m'].mean():,.2f} m, max distance={max_dist:,.2f} m, total signed USD={total_negative_signed:,.2f}, total abs USD={total_negative_abs:,.2f}"
    )


In [ ]:
# Display negative-threshold tables
for scenario in SCENARIOS_FOR_THRESHOLDS:
    if scenario not in negative_threshold_tables:
        continue
    print('\n' + '=' * 105)
    print(f'SCENARIO: {scenario.upper()} | Cumulative distance thresholds (negative avoided EAD assets)')
    print('=' * 105)
    display(
        negative_threshold_tables[scenario][[
            'Threshold_m', 'Assets_within_n', 'Assets_within_pct',
            'Negative_Avoided_EAD_within_USD_signed',
            'Negative_Avoided_EAD_within_USD_abs',
            'Negative_Avoided_EAD_within_abs_pct'
        ]]
    )
